# A ferramenta mlst de Torsten Seemann 
- Recebe seu genoma (.fna ou .fa) como entrada
- Localiza os 7 genes dentro do genoma usando BLAST (busca por similaridade de sequência)
- Compara cada gene com um banco de dados de referência (PubMLST), onde cada variante conhecida já tem um número — por exemplo, a versão 10 do gene adhP é simplesmente chamada de "alelo 10"
- Combina os 7 números (um por gene) para formar um perfil, ex: 10-1-2-1-3-2-2
- Consulta o banco para ver se esse perfil já existe — se sim, atribui o Sequence Type (ST) correspondente; se não, retorna ? (ST novo)

### Resultado final: cada genoma recebe um número de ST, como ST-17 ou ST-261.

In [ ]:
# Via conda (recomendado — gerencia dependências automaticamente)
conda create -n mlst_env -c conda-forge -c bioconda mlst
conda activate mlst_env

# Verificar instalação
mlst --version
mlst --list | grep sagalactiae   # confirmar esquema disponível

# Execução do MLST

In [ ]:
# Ativar ambiente
conda activate mlst_env

cd /mnt/dados/victor_baca/projeto_mlst

# Rodar mlst em todos os .fna e .fa do diretório
mlst --scheme sagalactiae genoma/*.fna genoma/*.fa > mlst_resultados.tsv

- O arquivo .tsv gerado tem o seguinte formato:

´´´
# Colunas do output:
```  

´´´
| FILE | SCHEME | ST | adhP | pheS | atr | glnA | sdhA | glcK | tkt |
| GCA_000007525.fna | sagalactiae | 17 | 3 | 3 | 6 | 1 | 1 | 4 | 3 |
| GCA_001045595.fa | sagalactiae | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |
| GCA_002085945.fna | sagalactiae | 260 | 2 | 6 | 4 | 9 | 3 | 2 | 8 |
```  



# Preparação da tabela de metadados integrada

In [1]:
# Script Python para integrar resultados
import pandas as pd

In [2]:
# 1. Carregar o TSV
mlst_raw = pd.read_csv('mlst_resultados.tsv', sep='\t', header=None,
                        names=['arquivo','esquema','ST',
                               'adhP','pheS','atr','glnA','sdhA','glcK','tkt'])

In [3]:
# 2. Criar genome_id a partir do caminho do arquivo
## genoma_id será o nome do arquivo sem a extensão e sem o caminho
mlst_raw['genome_id'] = (mlst_raw['arquivo']
                          .str.replace(r'^.*[\\/]', '', regex=True)      # remove pasta "genoma/"
                          .str.replace(r'_genomic\.(fna|fa)$', '', regex=True)  # remove sufixo
                          .str.replace(r'\.(fna|fa)$', '', regex=True))  # remove .fna/.fa residual

In [8]:
# Limpar alelos: "adhP(10)" → "10"
# for gene in ['adhP','pheS','atr','glnA','sdhA','glcK','tkt']:
    # mlst_raw[gene] = mlst_raw[gene].str.extract(r'\((\S+)\)')[0]

In [ ]:
# 3. Confirmar resultado
print(mlst_raw[['genome_id','ST','adhP','pheS']].to_string())
print(f"\nTotal de genomas : {len(mlst_raw)}")
print(f"STs únicos       : {mlst_raw['ST'].nunique()}")
print(f"STs novos (?)    : {(mlst_raw['ST'] == '?').sum()}")

In [6]:
# 3. Confirmar resultado
print(mlst_raw[['genome_id','ST']].to_string())
print(f"\nTotal de genomas : {len(mlst_raw)}")
print(f"STs únicos       : {mlst_raw['ST'].nunique()}")
print(f"STs novos (?)    : {(mlst_raw['ST'] == '?').sum()}")

                       genome_id    ST
0                          01173     7
1                    09mas018883     1
2                        138spar   261
3                          1B13M     1
4                           2-22   261
5                       2012-845    17
6                       32790-3A    17
7                       3966RFQB   167
8                          3B11V    23
9                          3B13R    23
10                         3B21M    23
11                         4B14M   130
12                         5B15M     1
13                         5B20M     1
14                         5B21V     1
15                         5B28V     1
16                          5B2M     1
17                         6B19M     1
18                         7B18M     -
19                        874391    17
20                         8B14M   103
21                A909_Cas9_AEKO     7
22       A909_Cas9_AEKOrevertant     7
23                    A909_dCas9     7
24                    A90